# 11.5 — The Delta Transaction Log

**Chapter 11, §6.1 (lines 519–548 and Figure 11.7) and §6.2 (lines 636–644).**

The chapter's claim is that a Delta table is not the directory it lives in. It is the ordered log
of commits in `_delta_log/`, and the Parquet files underneath are only the files that log
currently happens to name. Everything the format offers — atomic commits, snapshot isolation,
time travel, safe deletion — follows from that one change of definition.

This notebook makes three commits to a tiny table and reads the log after each one.

> **On the engine.** Delta Lake's Spark connector is published for Spark 4.0 and does not load on
> the Spark 4.2 this course uses: `io.delta:delta-spark_2.13:4.0.0` fails with
> `NoClassDefFoundError: org/apache/spark/sql/catalyst/expressions/TimeAdd`. The notebook
> therefore uses **`deltalake`, the Rust implementation of the same open protocol**, which writes
> the identical on-disk format. That is not a workaround so much as the point of §6.3: the table
> is a specification, not an engine's private storage.

In [1]:
import json, os, shutil, tempfile
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch11-delta")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH)
TABLE = os.path.join(SCRATCH, "trips")
LOG = os.path.join(TABLE, "_delta_log")


def log_files():
    return sorted(os.listdir(LOG))


def data_files():
    return sorted(f for f in os.listdir(TABLE) if f.endswith(".parquet"))


def commit(version):
    # Every line of a commit file is one action.
    path = os.path.join(LOG, f"{version:020d}.json")
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

## Commit 0 — creating the table

In [2]:
write_deltalake(TABLE, pa.table({
    "trip_id": [1, 2, 3, 4],
    "city":    ["BOS", "BOS", "NYC", "NYC"],
    "fare":    [9.5, 12.0, 7.25, 30.0],
}))

print("log  :", log_files())
print("data :", len(data_files()), "parquet file(s)")
print("actions in commit 0:", [list(a)[0] for a in commit(0)])

log  : ['00000000000000000000.json']
data : 1 parquet file(s)
actions in commit 0: ['commitInfo', 'protocol', 'metaData', 'add']


Four actions. Three of them are the ones the chapter names: `protocol` says which reader and
writer versions are required, `metaData` carries the schema and the partition columns, and `add`
introduces a data file. The fourth, `commitInfo`, is provenance — what operation ran, when, and
which engine wrote it.

`add` is the interesting one. It is not just a file name.

In [3]:
add = [a["add"] for a in commit(0) if "add" in a][0]
print("path :", add["path"])
print("size :", add["size"], "bytes")
print("stats:", json.dumps(json.loads(add["stats"]), indent=2))

path : part-00000-e6a78181-4af7-4a66-a1f9-47bb6f813135-c000.snappy.parquet
size : 1127 bytes
stats: {
  "numRecords": 4,
  "minValues": {
    "trip_id": 1,
    "fare": 7.25,
    "city": "BOS"
  },
  "maxValues": {
    "trip_id": 4,
    "city": "NYC",
    "fare": 30.0
  },
  "nullCount": {
    "city": 0,
    "fare": 0,
    "trip_id": 0
  }
}


Row count, per-column minima and maxima, per-column null counts — recorded **in the log**, at
commit time. A query for `city = 'SFO'` can be answered against this file without opening it,
because the log already says the largest city value in it is `NYC`. That is the "granular
per-file column statistics ... without requiring physical directory partitioning" of §3.3's last
paragraph, and it is why 11.3's high-cardinality partition trick is not the only way to skip
data.

## Commit 1 — an append

In [4]:
write_deltalake(TABLE, pa.table({
    "trip_id": [5, 6],
    "city":    ["SFO", "BOS"],
    "fare":    [11.0, 8.0],
}), mode="append")

print("log  :", log_files())
print("actions in commit 1:", [list(a)[0] for a in commit(1)])
print("adds :", [a["add"]["path"] for a in commit(1) if "add" in a])

log  : ['00000000000000000000.json', '00000000000000000001.json']
actions in commit 1: ['commitInfo', 'add']
adds : ['part-00000-f0d532fa-5ba0-48f7-96c5-fe2be004b42e-c000.snappy.parquet']


One new commit file naming one new data file. Nothing that commit 0 wrote was touched — no file
was rewritten, no directory was renamed, and a reader that started before this commit is still
reading a consistent table.

## Commit 2 — a delete

In [5]:
dt = DeltaTable(TABLE)
metrics = dt.delete("city = 'BOS'")
print({k: metrics[k] for k in ("num_removed_files", "num_added_files", "num_deleted_rows")})

print("log  :", log_files())
print("actions in commit 2:", [list(a)[0] for a in commit(2)])
for action in commit(2):
    if "remove" in action:
        print("  remove:", action["remove"]["path"], "dataChange =", action["remove"]["dataChange"])
    if "add" in action:
        print("  add   :", action["add"]["path"])

{'num_removed_files': 2, 'num_added_files': 1, 'num_deleted_rows': 3}
log  : ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
actions in commit 2: ['commitInfo', 'add', 'remove', 'remove']
  add   : part-00000-a639622a-a73f-4613-b74a-63368cd11198-c000.zstd.parquet
  remove: part-00000-f0d532fa-5ba0-48f7-96c5-fe2be004b42e-c000.snappy.parquet dataChange = True
  remove: part-00000-e6a78181-4af7-4a66-a1f9-47bb6f813135-c000.snappy.parquet dataChange = True


Deleting rows from immutable files means writing new files without them and **logging the old
ones as removed**. `remove` does not delete anything from storage; it takes the file out of the
table.

## The table is the log, not the directory

In [6]:
dt = DeltaTable(TABLE)
print("parquet files in the directory :", len(data_files()))
print("files the current version uses :", len(dt.file_uris()))
print("rows in the current version    :", dt.to_pyarrow_table().num_rows)
dt.to_pandas().sort_values("trip_id").reset_index(drop=True)

parquet files in the directory : 3
files the current version uses : 1
rows in the current version    : 3


,trip_id,city,fare
0,3,NYC,7.25
1,4,NYC,30.00
2,5,SFO,11.00


More files on disk than in the table. The difference is exactly the history: those files are not
garbage, they are what earlier versions are made of.

## Time travel

In [7]:
for v in range(dt.version() + 1):
    snap = DeltaTable(TABLE, version=v)
    rows = snap.to_pyarrow_table()
    print(f"version {v}: {rows.num_rows} rows, {len(snap.file_uris())} file(s), "
          f"cities {sorted(set(rows.column('city').to_pylist()))}")

version 0: 4 rows, 1 file(s), cities ['BOS', 'NYC']
version 1: 6 rows, 2 file(s), cities ['BOS', 'NYC', 'SFO']
version 2: 3 rows, 1 file(s), cities ['NYC', 'SFO']


In [8]:
for entry in reversed(dt.history()):
    op = entry["operation"]
    m = entry.get("operationMetrics", {})
    print(f"v{entry['version']}  {op:<8} "
          f"added {m.get('num_added_files', 0)} removed {m.get('num_removed_files', 0)}")

v0  WRITE    added 1 removed 0
v1  WRITE    added 1 removed 0
v2  DELETE   added 1 removed 2


The version numbers are the commit-file numbers; there is no separate versioning mechanism. A
timestamp-based query resolves to one of these versions the same way.

## Checkpoints

Reconstructing the current table means replaying every commit from zero, which stops being cheap
after a few thousand of them. The chapter's answer is the periodic Parquet checkpoint that
consolidates the replayed state.

In [9]:
for i in range(7, 16):
    write_deltalake(TABLE, pa.table({"trip_id": [i * 10], "city": ["BOS"], "fare": [float(i)]}),
                    mode="append")

dt = DeltaTable(TABLE)
dt.create_checkpoint()
print("version:", dt.version())
print([f for f in log_files() if not f.endswith(".json")])
print(json.load(open(os.path.join(LOG, "_last_checkpoint"))))

version: 11
['00000000000000000011.checkpoint.parquet', '_last_checkpoint']
{'version': 11, 'size': 14, 'sizeInBytes': 16657, 'numOfAddFiles': 10}


`_last_checkpoint` tells a reader where to start: load that Parquet file, then replay only the
JSON commits written after it. The chapter's Figure 11.7 draws exactly this — a run of numbered
JSON files with a checkpoint sitting among them.

## Cleaning up, and why it is dangerous

`VACUUM` is what actually deletes bytes. Its safety is entirely in its retention period, and the
chapter's seven-day default is what stops it from deleting the files that time travel — or a
query that is still running — depends on.

In [10]:
print("removable with the default retention:",
      len(dt.vacuum(dry_run=True)), "files")
print("removable if retention is dropped to zero:",
      len(dt.vacuum(retention_hours=0, dry_run=True, enforce_retention_duration=False)), "files")

removable with the default retention: 0 files
removable if retention is dropped to zero: 2 files


Nothing is old enough to remove under the default. Lower the guard and the files that only
history still references become candidates — after which `VERSION AS OF 0` has metadata pointing
at files that are no longer there. That is the distinction Exercise 6(a) asks for: expiring a
snapshot removes the *metadata* that identifies a version, while vacuuming removes the *data
files* that version was made of, and a time-travel query can fail from either.

## Conclusion

* A Delta commit is one JSON file of actions. `add` introduces a file and carries its row count
  and per-column statistics; `remove` takes a file out of the table without deleting it;
  `metaData` and `protocol` carry the schema and the compatibility requirements.
* Data files are never modified. A delete is new files plus `remove` actions, which is why the
  directory accumulates more Parquet than the current version uses.
* Every version is still addressable because the commits that built it are still on disk, and a
  checkpoint exists only to save a reader from replaying all of them.
* The commit is a single atomic file creation — the flat key space's one atomic primitive, from
  11.1, used exactly once per transaction.